### What it does
Script to handle words with "-", for example "risk-taker". The aim is to modify it into "risk taker", "risktaker". 
It searchs "-" in words ignoring case. Then it split the words by "-" and joint the words in one taking into account lowcase letter.
It creates new table with the the same columns as original table. Original table with new one are merged and sorted by lemma. 

In [62]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# pandas library to work with tables/dataframes
import pandas as pd


# Regular expression library
import re
import ast

In [63]:
#file name
file_name = "rules_08MAR_English"

In [64]:
#read csv file to the pandas table
df_rules = pd.read_csv(file_name+".csv")

In [65]:
#get info about the table
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 22 columns):
Lemma                     1202 non-null object
Word_Type                 1202 non-null object
Priority                  686 non-null object
Category                  1202 non-null object
Primary_subcategory       1202 non-null object
Secondary_subcategory     293 non-null object
Alt_Field                 380 non-null object
Alt_Sg_Replacement        609 non-null object
Alt_Sg_people_first       132 non-null object
Alt_Sg_reframed           226 non-null object
Alt_Pl_collective_noun    39 non-null object
Alt_Pl                    121 non-null object
Alt_Pl_people_first       22 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
3rd_party_source          162 non-null object
Sample_Sentences          5 non-null object
False_Positives           14 non-null object
Notes                     254 non-null object
ToClarify               

In [66]:
#first 5 row of the table
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,3rd_party_source,Sample_Sentences,False_Positives,Notes,ToClarify,Unnamed: 20,Unnamed: 21
0,attention-seeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,...,NaN,NaN,NaN,NaN,NaN,NaN,Can refer to people with mental or psychiatric...,NaN,NaN,NaN
1,beautiful,a,NaN,gendered,female_stereotype,NaN,NaN,attractive | good-looking,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bossy,a,NaN,gendered,female_stereotype,NaN,NaN,in charge | supervising | responsible for resu...,NaN,NaN,...,NaN,NaN,NaN,70 Inclusive Language Principles That Will Mak...,NaN,NaN,This term has been skewed to be a negative ter...,NaN,NaN,NaN
3,heroine,s,1,gendered,female_stereotype,NaN,NaN,person of the hour | person making sacrifices ...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,histrionic,s,23-12-2021,gendered,female_stereotype,NaN,NaN,NaN,person with HPD | person with histrionic perso...,lively | dramatic | vivacious | enthusiastic |...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Remove unnecessary columns

In [67]:
#create a subset of dataframe from the 'Lemma' till the last column with alternatives '3rd_party_alternatives'
df_rules=df_rules.loc[:, 'Lemma':'3rd_party_alternatives']

In [68]:
#check new table
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 15 columns):
Lemma                     1202 non-null object
Word_Type                 1202 non-null object
Priority                  686 non-null object
Category                  1202 non-null object
Primary_subcategory       1202 non-null object
Secondary_subcategory     293 non-null object
Alt_Field                 380 non-null object
Alt_Sg_Replacement        609 non-null object
Alt_Sg_people_first       132 non-null object
Alt_Sg_reframed           226 non-null object
Alt_Pl_collective_noun    39 non-null object
Alt_Pl                    121 non-null object
Alt_Pl_people_first       22 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 141.0+ KB


In [69]:
#check fisrts 5 rows of the table
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
0,attention-seeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN
1,beautiful,a,NaN,gendered,female_stereotype,NaN,NaN,attractive | good-looking,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bossy,a,NaN,gendered,female_stereotype,NaN,NaN,in charge | supervising | responsible for resu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,heroine,s,1,gendered,female_stereotype,NaN,NaN,person of the hour | person making sacrifices ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,histrionic,s,23-12-2021,gendered,female_stereotype,NaN,NaN,NaN,person with HPD | person with histrionic perso...,lively | dramatic | vivacious | enthusiastic |...,NaN,NaN,NaN,NaN,NaN


In [70]:
#check the number of columns
len(list(df_rules.columns))

15

### Remove white spaces from Category and Primary_subcategory

In [71]:
df_rules['Primary_subcategory'] = df_rules['Primary_subcategory'].str.strip()

In [72]:
df_rules['Category'] = df_rules['Category'].str.strip()

### Splitting and merging words by hyphen

In [73]:
list_word = []

list_word_type = []
list_priority = []
list_category = []
list_primary_subcategory = []
list_secondary_subcategory = []
list_alt_field= []
list_alt_sg_replacement = []
list_alt_sg_people_first = []
list_alt_sg_reframed = []
list_alt_pl_collective_noun = []
list_alt_pl = []
list_alt_pl_people_first = []
list_alt_pl_reframed = []
list_3rd_party_alternatives = []


for index, row in df_rules.iterrows():
    word = row["Lemma"]
    span = re.search("-", word)
    #only merge and split words if they are two, for example risk-taker (two words: risk and taker)
    if type(span) == re.Match and len(word.rsplit("-"))==2:
        #merge words by "-"
        list_word.append(word.rsplit("-")[0]+word.rsplit("-")[1].lower())
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_alt_pl_reframed.append(row['Alt_Pl_reframed'])
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        
        
        
        #split words by "-"  
        list_word.append(word.rsplit("-")[0]+" "+word.rsplit("-")[1])
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_alt_pl_reframed.append(row['Alt_Pl_reframed'])
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        

### Created new dataframe from lists

In [74]:
#NEW  from the bunch of created lists create a dataframe
data = pd.DataFrame(list(zip(list_word,list_word_type,list_priority, list_category, list_primary_subcategory, list_secondary_subcategory, 
list_alt_field, list_alt_sg_replacement, list_alt_sg_people_first, list_alt_sg_reframed, list_alt_pl_collective_noun,
list_alt_pl, list_alt_pl_people_first, list_alt_pl_reframed, list_3rd_party_alternatives)), columns=list(df_rules.columns))

In [75]:
# check number of columns in the new dataframe
len(data.columns)

15

In [76]:
#list of names of the columns
data_columns = list(data.columns)
df_rules_columns = list(df_rules.columns)

### Test name of the columns

In [77]:
for name1, name2 in zip(df_rules_columns, data_columns):
    if name1!=name2:
        print(name1, name2)
    else:
        print("correct")

correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct


In [78]:
#check first 5 rows of the new dataframe
data.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
0,attentionseeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN
1,attention seeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN
2,everyman,s,NaN,gendered,function,NaN,NaN,everybody | all,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,every man,s,NaN,gendered,function,NaN,NaN,everybody | all,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gunman,s,NaN,gendered,function,NaN,NaN,shooter | they,NaN,NaN,NaN,shooters,NaN,NaN,NaN


In [79]:
#check the last 5 rows of the new dataframe
data.tail()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
251,blind sided,a,1,unconscious_bias,vision,cultural,NaN,awestruck | astounded | bewildered | confused ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
252,doubleblind review,a+s,1,unconscious_bias,vision,cultural,NaN,anonymous review | identity-hidden review | do...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
253,double blind review,a+s,1,unconscious_bias,vision,cultural,NaN,anonymous review | identity-hidden review | do...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254,eyeopener,s,1,unconscious_bias,vision,cultural,NaN,utterly shocking | real surprise,NaN,"startling (situation, revelation, ...)",NaN,NaN,NaN,NaN,NaN
255,eye opener,s,1,unconscious_bias,vision,cultural,NaN,utterly shocking | real surprise,NaN,"startling (situation, revelation, ...)",NaN,NaN,NaN,NaN,NaN


### New created dataframe

In [80]:
# check info about new dataframe
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256 entries, 0 to 255
Data columns (total 15 columns):
Lemma                     256 non-null object
Word_Type                 256 non-null object
Priority                  114 non-null object
Category                  256 non-null object
Primary_subcategory       256 non-null object
Secondary_subcategory     54 non-null object
Alt_Field                 100 non-null object
Alt_Sg_Replacement        110 non-null object
Alt_Sg_people_first       26 non-null object
Alt_Sg_reframed           72 non-null object
Alt_Pl_collective_noun    8 non-null object
Alt_Pl                    18 non-null object
Alt_Pl_people_first       6 non-null object
Alt_Pl_reframed           0 non-null float64
3rd_party_alternatives    0 non-null float64
dtypes: float64(2), object(13)
memory usage: 30.1+ KB


In [81]:
#original dataframe
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 15 columns):
Lemma                     1202 non-null object
Word_Type                 1202 non-null object
Priority                  686 non-null object
Category                  1202 non-null object
Primary_subcategory       1202 non-null object
Secondary_subcategory     293 non-null object
Alt_Field                 380 non-null object
Alt_Sg_Replacement        609 non-null object
Alt_Sg_people_first       132 non-null object
Alt_Sg_reframed           226 non-null object
Alt_Pl_collective_noun    39 non-null object
Alt_Pl                    121 non-null object
Alt_Pl_people_first       22 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 141.0+ KB


### Merge two dataframes (original and new one)

In [82]:
#merge two dataframes
df_fin = df_rules.append(data, ignore_index = True)

In [83]:
#check info about merged dataframe
df_fin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 15 columns):
Lemma                     1458 non-null object
Word_Type                 1458 non-null object
Priority                  800 non-null object
Category                  1458 non-null object
Primary_subcategory       1458 non-null object
Secondary_subcategory     347 non-null object
Alt_Field                 480 non-null object
Alt_Sg_Replacement        719 non-null object
Alt_Sg_people_first       158 non-null object
Alt_Sg_reframed           298 non-null object
Alt_Pl_collective_noun    47 non-null object
Alt_Pl                    139 non-null object
Alt_Pl_people_first       28 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 171.0+ KB


### Sorted rows in dataframe by lemmas (in alphabetic order)

In [84]:
df_sorted = df_fin.sort_values(by=['Lemma'], inplace=False, ascending=True)

In [85]:
df_sorted.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1458 entries, 579 to 1091
Data columns (total 15 columns):
Lemma                     1458 non-null object
Word_Type                 1458 non-null object
Priority                  800 non-null object
Category                  1458 non-null object
Primary_subcategory       1458 non-null object
Secondary_subcategory     347 non-null object
Alt_Field                 480 non-null object
Alt_Sg_Replacement        719 non-null object
Alt_Sg_people_first       158 non-null object
Alt_Sg_reframed           298 non-null object
Alt_Pl_collective_noun    47 non-null object
Alt_Pl                    139 non-null object
Alt_Pl_people_first       28 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 182.2+ KB


### Remove must have from file as false positive

In [86]:
df_sorted[df_sorted["Lemma"]=="must have"]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
1357,must have,s,1,style,exaggerating,NaN,NaN,requirement | criterion | essential | quality ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1355,must have,a,1,style,exaggerating,NaN,NaN,essential | helpful | valuable | mission-criti...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [87]:
#get index of false positives
i = df_sorted[((df_sorted.Lemma == 'must have'))].index

In [88]:
#drop rows with false positives
df_sorted = df_sorted.drop(i)

### Remove binary_pronouns  from file

In [91]:
#get index of binary_pronouns
i_binary = df_sorted[((df_sorted.Primary_subcategory == 'binary_pronouns'))].index

In [92]:
#drop rows with binary pronouns
df_sorted = df_sorted.drop(i_binary)

### Save the sorted dataframe to csv

In [93]:
#save to csv file after preprocessing
df_sorted.to_csv(file_name+"_h"+".csv", index=False)

In [95]:
#df_sorted["Primary_subcategory"].unique()

In [96]:
#test read of dataframe
df_test = pd.read_csv(file_name+"_h"+".csv")

In [97]:
#check info
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1450 entries, 0 to 1449
Data columns (total 15 columns):
Lemma                     1450 non-null object
Word_Type                 1450 non-null object
Priority                  798 non-null object
Category                  1450 non-null object
Primary_subcategory       1450 non-null object
Secondary_subcategory     347 non-null object
Alt_Field                 480 non-null object
Alt_Sg_Replacement        711 non-null object
Alt_Sg_people_first       158 non-null object
Alt_Sg_reframed           298 non-null object
Alt_Pl_collective_noun    47 non-null object
Alt_Pl                    139 non-null object
Alt_Pl_people_first       28 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 170.0+ KB


In [98]:
#check first 5 rows 
df_test.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
0,ADD,abbr,11-02-2022,style,abbreviation,NaN,NaN,attention deficit disorder,person with ADD | person with attention defici...,easily bored | inattentive | easily distracted...,NaN,NaN,NaN,NaN,NaN
1,ADHD,abbr,11-02-2022,style,abbreviation,NaN,NaN,attention deficit hyperactivity disorder,person with ADHD | person with attention defic...,fidgety | restless | easily bored,NaN,NaN,NaN,NaN,NaN
2,ASD,abbr,11-02-2022,style,abbreviation,NaN,autism spectrum disorder,neurodiversity,person with ASD | person with autism sprectru...,neurodevelopmental disorder characterized by d...,NaN,NaN,NaN,NaN,NaN
3,ASL,abbr,11-02-2022,style,abbreviation,NaN,NaN,American Sign Language,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,African queen,s,NaN,openly_discriminating,homophobia,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
